# Batch2 평가 데이터 추출

Batch1에서 확인한 구조를 기준으로 **형식 확인 → 같은 형식으로 추출 → 저장 결과 검증**을 수행한다. 값의 상관관계나 모델 선택은 수행하지 않는다. 실행 위치는 `notebooks/`다.

Batch2에는 실제 cycle 1 profile이 있고 일부 `cycle_life`는 결측이다. Batch1의 dummy 제외 규칙과 기록 길이 관계를 일괄 적용하지 않고 원본 값을 보존한다. 모델링에서는 Batch1으로 변수·모델을 고정한 뒤 평가에 사용한다.

In [1]:
# Batch1의 구조를 기준으로 삼고, Batch2의 저장 파일은 배치별로 구분합니다.
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

batch1_path = DATA_DIR / "Batch1.mat"
batch_paths = {"batch2": DATA_DIR / "Batch2.mat"}
assert all(path.is_file() for path in [batch1_path, *batch_paths.values()])

## Batch1과 비교할 구조

최상위 키, `batch`의 필드와 reference 형태, `summary`·`cycles`의 필드, `channel_id`의 여섯 요소, `Vdlin` 전압축을 비교합니다. 셀 수와 cycle 수는 배치마다 다를 수 있으므로 동일한 수를 요구하지 않습니다. 상세한 값 출력과 EDA는 Batch1에서 했던 작업을 반복하지 않습니다.

In [2]:
# 구조 기준만 읽습니다. Batch1을 다시 추출하거나 저장하지 않습니다.
with h5py.File(batch1_path, "r") as f:
    batch = f["batch"]
    batch1_root_fields = set(f.keys())
    batch1_fields = set(batch.keys())
    summary_fields = list(f[batch["summary"][0, 0]].keys())
    cycle_fields = list(f[batch["cycles"][0, 0]].keys())
    reference_vdlin = f[batch["Vdlin"][0, 0]][()].ravel()

summary_columns = ["cell_id", "cycle", *[key for key in summary_fields if key != "cycle"]]
profile_fields = {"Qdlin": "qdlin", "Tdlin": "tdlin", "discharge_dQdV": "dqdv"}
print("비교 기준:", len(batch1_fields), "개 batch 필드 /", len(reference_vdlin), "개 전압점")

비교 기준: 8 개 batch 필드 / 1000 개 전압점


## 추출 및 저장 규칙

`cell_id`는 1부터 시작한다. CSV와 NPZ의 셀 번호를 맞추고 `cell_N_cycle`에는 profile 각 행의 실제 cycle 번호를 저장한다. 병합 키는 `(batch, cell_id)`다.

수명 결측은 빈 값으로 보존한다. 첫 cycle은 모든 시계열 필드가 길이 2의 `[0, 0]`일 때만 dummy로 제외하며 실제 cycle 1은 유지한다. 내부저항 0값과 원본 정책 문자열도 유지한다.

In [3]:
# MATLAB char만 문자로 해석하고, channel_id 같은 숫자 배열은 그대로 읽습니다.
def read_h5_value(f, ref):
    obj = f[ref]
    value = obj[()].squeeze()
    if obj.attrs.get("MATLAB_class") == b"char":
        return "".join(chr(int(code)) for code in np.atleast_1d(value) if code != 0)
    return value.item() if value.ndim == 0 else value


def extract_batch(mat_path, batch_name):
    """Batch1과 구조를 비교한 뒤 셀·summary·profile·전압축 파일을 저장합니다."""
    cell_rows, summary_rows, profiles = [], [], {}
    dummy_cells = after_life_cells = profile_nonfinite = 0

    with h5py.File(mat_path, "r") as f:
        assert set(f.keys()) == batch1_root_fields, f"{batch_name}: 최상위 구조가 다릅니다"
        batch = f["batch"]
        assert set(batch.keys()) == batch1_fields, f"{batch_name}: batch 필드가 다릅니다"
        n_cells = batch["cycle_life"].shape[0]
        assert all(
            batch[key].shape == (n_cells, 1)
            and h5py.check_dtype(ref=batch[key].dtype) is not None
            for key in batch1_fields
        ), f"{batch_name}: cell-level reference 구조가 다릅니다"

        for cell_idx in range(n_cells):
            cell_id = cell_idx + 1  # HDF5 인덱스와 저장용 셀 번호를 구분합니다.
            summary = f[batch["summary"][cell_idx, 0]]
            cycles = f[batch["cycles"][cell_idx, 0]]
            assert set(summary.keys()) == set(summary_fields), (batch_name, cell_id, "summary 필드")
            assert set(cycles.keys()) == set(cycle_fields), (batch_name, cell_id, "cycles 필드")

            values = {key: summary[key][()].ravel() for key in summary_fields}
            cycle_numbers = values["cycle"]
            n_cycles = len(cycle_numbers)
            assert np.array_equal(cycle_numbers, np.arange(1, n_cycles + 1)), (batch_name, cell_id, "cycle 번호")
            assert all(summary[key].shape == (1, n_cycles) for key in summary_fields), (batch_name, cell_id, "summary 길이")
            assert all(
                cycles[key].shape == (n_cycles, 1)
                and h5py.check_dtype(ref=cycles[key].dtype) is not None
                for key in cycle_fields
            ), (batch_name, cell_id, "cycle reference 구조")
            vdlin = read_h5_value(f, batch["Vdlin"][cell_idx, 0])
            assert np.array_equal(vdlin, reference_vdlin), (batch_name, cell_id, "Vdlin 전압축")

            channel = read_h5_value(f, batch["channel_id"][cell_idx, 0])
            assert channel.shape == (6,), (batch_name, cell_id, "channel_id 구조")
            policy = read_h5_value(f, batch["policy_readable"][cell_idx, 0])
            assert isinstance(policy, str), (batch_name, cell_id, "policy 문자열")
            life = read_h5_value(f, batch["cycle_life"][cell_idx, 0])
            cell_rows.append({"cell_id": cell_id, "channel": int(channel[4]), "policy": policy, "cycle_life": life})
            after_life_cells += int(np.isfinite(life) and cycle_numbers[-1] > life)

            frame = pd.DataFrame(values)
            frame.insert(0, "cell_id", cell_id)
            frame["cycle"] = frame["cycle"].astype(int)
            summary_rows.append(frame[summary_columns])

            # Batch1의 첫 행 제외를 그대로 적용하지 않고, 실제 dummy인지 판단합니다.
            is_dummy = all(
                np.array_equal(f[cycles[key][0, 0]][()].ravel(), [0, 0])
                for key in cycle_fields
            )
            start = int(is_dummy)
            dummy_cells += start
            profiles[f"cell_{cell_id}_cycle"] = cycle_numbers[start:].astype(int)
            for source, target in profile_fields.items():
                array = np.stack([
                    f[ref][()].ravel() for ref in cycles[source][start:, 0]
                ])
                assert array.shape == (n_cycles - start, len(reference_vdlin)), (batch_name, cell_id, source)
                profile_nonfinite += int(array.size - np.isfinite(array).sum())
                profiles[f"cell_{cell_id}_{target}"] = array

    # 수명 결측을 허용하는 정수형으로 저장하고, 식별용 channel이 셀마다 다른지 확인합니다.
    cell_table = pd.DataFrame(cell_rows)
    cell_table["cycle_life"] = cell_table["cycle_life"].astype("Int64")
    assert cell_table["channel"].nunique() == n_cells, f"{batch_name}: channel 중복"
    summary_table = pd.concat(summary_rows, ignore_index=True)
    cell_table.to_csv(PROCESSED_DIR / f"{batch_name}_cells.csv", index=False)
    summary_table.to_csv(PROCESSED_DIR / f"{batch_name}_cycle_summary.csv", index=False)
    pd.DataFrame({"Vdlin": reference_vdlin}).to_csv(PROCESSED_DIR / f"{batch_name}_vdlin.csv", index=False)

    # 압축 저장이 완료된 파일만 최종 이름으로 바꿔 중간에 끊긴 NPZ를 사용하지 않게 합니다.
    print(f"{batch_name}: 구조 확인 완료, profile 압축 저장 중", flush=True)
    profiles_path = PROCESSED_DIR / f"{batch_name}_cycle_profiles.npz"
    temporary_path = profiles_path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary_path, **profiles)
    temporary_path.replace(profiles_path)
    return {
        "batch": batch_name, "cells": n_cells, "summary_rows": len(summary_table),
        "missing_cycle_life": int(cell_table["cycle_life"].isna().sum()),
        "dummy_cells": dummy_cells, "last_cycle_after_life": after_life_cells,
        "profile_nonfinite": profile_nonfinite,
    }

In [4]:
# Batch1에서 한 상세 탐색을 반복하지 않고, 구조 검사와 저장만 배치별로 실행합니다.
extraction_results = []
for batch_name, mat_path in batch_paths.items():
    result = extract_batch(mat_path, batch_name)
    extraction_results.append(result)
    print(f"{batch_name}: 네 파일 저장 완료", flush=True)

print(pd.DataFrame(extraction_results).to_string(index=False))

batch2: 구조 확인 완료, profile 압축 저장 중


batch2: 네 파일 저장 완료


 batch  cells  summary_rows  missing_cycle_life  dummy_cells  last_cycle_after_life  profile_nonfinite
batch2     47         26904                   8            0                     39                  0


## 저장 결과 확인

저장된 파일을 다시 읽어 CSV 병합에서 행이 빠지지 않는지, NPZ의 셀 번호·cycle 번호·1000-point 전압축이 맞는지 검사합니다. 각 셀의 첫 profile을 원본 `.mat`과 대조해 실제 cycle 1이 누락되지 않았는지도 확인합니다. 이후 EDA에서는 수명 결측과 기록 기간의 차이를 별도로 해석해야 합니다.

In [5]:
# 원본과 저장 파일을 대조하는 한 번의 검사로 평가 데이터의 입력 연결을 확인합니다.
for batch_name, mat_path in batch_paths.items():
    cells = pd.read_csv(PROCESSED_DIR / f"{batch_name}_cells.csv")
    summary = pd.read_csv(PROCESSED_DIR / f"{batch_name}_cycle_summary.csv")
    vdlin = pd.read_csv(PROCESSED_DIR / f"{batch_name}_vdlin.csv")["Vdlin"].to_numpy()
    assert cells.columns.tolist() == ["cell_id", "channel", "policy", "cycle_life"]
    assert summary.columns.tolist() == summary_columns
    assert cells["cell_id"].tolist() == list(range(1, len(cells) + 1))
    assert len(summary.merge(cells, on="cell_id", validate="many_to_one")) == len(summary)
    # CSV 소수점 변환에서 생기는 작은 오차는 허용합니다.
    assert np.allclose(vdlin, reference_vdlin, rtol=0, atol=1e-12)
    assert not summary.duplicated(["cell_id", "cycle"]).any()

    with h5py.File(mat_path, "r") as f, np.load(PROCESSED_DIR / f"{batch_name}_cycle_profiles.npz") as saved:
        assert set(saved.files) == {
            f"cell_{cell_id}_{field}"
            for cell_id in cells["cell_id"] for field in ("cycle", *profile_fields.values())
        }
        for cell_id, group in summary.groupby("cell_id", sort=True):
            cycle = saved[f"cell_{cell_id}_cycle"]
            assert np.array_equal(cycle, group["cycle"].to_numpy())  # Batch2는 첫 cycle도 실제 데이터입니다.
            original = f[f["batch"]["cycles"][cell_id - 1, 0]]
            for source, target in profile_fields.items():
                array = saved[f"cell_{cell_id}_{target}"]
                assert array.shape == (len(cycle), len(vdlin))
                assert np.array_equal(array[0], f[original[source][0, 0]][()].ravel(), equal_nan=True)
    print(f"{batch_name}: {len(cells)}개 셀의 ID·cycle·profile 저장 검증 완료")

batch2: 47개 셀의 ID·cycle·profile 저장 검증 완료


## 평가로 전달할 파일

```text
data/processed/
├── batch2_cells.csv
├── batch2_cycle_summary.csv
├── batch2_cycle_profiles.npz
└── batch2_vdlin.csv
```

파일 형식과 열 이름은 Batch1과 같다. 원본 raw 시계열은 필요할 때 `.mat`에서 읽는다. 큰 원본과 추출 파일은 Git에서 제외한다.

수명 결측을 보존하고 기록 길이를 수명으로 대체하지 않는다. 변수 생성식과 모델·파라미터는 Batch1에서 고정하며 이 데이터는 최종 평가에 전달한다.